<a href="https://colab.research.google.com/github/aadithya-vimal/SamsungInnovationCampus/blob/main/Airline_passenger_satisfaction_29_4_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Airline Passenger Satisfaction — Segmentation and Prediction

## Introduction

This notebook analyzes survey data from airline passengers to understand what drives satisfaction and identify at-risk customer segments.

The analysis is structured in four parts:
- **Part 1** covers exploratory data analysis and preprocessing — understanding the data, identifying satisfaction drivers, and preparing features for modelling.
- **Part 2** applies K-Means clustering to segment passengers into distinct groups based on their service ratings.
- **Part 3** builds and evaluates two classifiers — Random Forest and XGBoost — to predict passenger satisfaction, using cluster labels as an additional feature.
- **Part 4** translates model outputs into actionable business strategy, identifying the highest-risk customer segment and recommending targeted service improvements.

The dataset contains survey responses from airline passengers including demographic information, service ratings across 14 categories, and an overall satisfaction label. The training set contains 103,904 passengers and the test set contains 25,976 passengers.

# Part 1: Exploratory Data Analysis and Preprocessing



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import os

df=pd.read_csv('/content/train.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/train.csv'

In [ ]:
# Calculate overall satisfaction rate
satisfaction_counts = df['satisfaction'].value_counts()
total_passengers = len(df)

satisfied_rate = satisfaction_counts.get('satisfied', 0) / total_passengers * 100
neutral_dissatisfied_rate = satisfaction_counts.get('neutral or dissatisfied', 0) / total_passengers * 100

print(f"\nOverall Satisfaction Rate:")
print(f"Satisfied: {satisfied_rate:.2f}%")
print(f"Neutral or Dis_satisfied: {neutral_dissatisfied_rate:.2f}%")

Now, let's compare the average ratings for service categories between satisfied and neutral/dissatisfied customers to understand the drivers of satisfaction. We'll identify the columns related to service ratings and then calculate their means based on satisfaction levels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Identify service-related columns
service_columns = ['Inflight wifi service', 'Departure/Arrival time convenient',
                   'Ease of Online booking', 'Gate location', 'Food and drink',
                   'Online boarding', 'Seat comfort', 'Inflight entertainment',
                   'On-board service', 'Leg room service', 'Baggage handling',
                   'Checkin service', 'Inflight service', 'Cleanliness']

# Group by satisfaction and calculate the mean for service categories
service_ratings_comparison = df.groupby('satisfaction')[service_columns].mean().transpose()

print("Average service ratings by satisfaction level:")
display(service_ratings_comparison)

# Create a visual to show the differences
fig, ax = plt.subplots(figsize=(12, 8))
service_ratings_comparison.plot(kind='bar', ax=ax)
plt.title('Average Service Ratings by Passenger Satisfaction')
plt.xlabel('Service Category')
plt.ylabel('Average Rating')
plt.xticks(rotation=90)
plt.legend(title='Satisfaction')
plt.tight_layout()
plt.show()

Next, let's analyze the relationship between customer type (Loyal vs. Disloyal) and satisfaction.

In [ ]:
# Analyze the relationship between customer type and satisfaction
customer_type_satisfaction = pd.crosstab(df['Customer Type'], df['satisfaction'], normalize='index') * 100

print("Relationship between Customer Type and Satisfaction (%):")
display(customer_type_satisfaction)

# Visualize the relationship
fig, ax = plt.subplots(figsize=(8, 6))
customer_type_satisfaction.plot(kind='bar', stacked=True, ax=ax)
plt.title('Satisfaction Rate by Customer Type')
plt.xlabel('Customer Type')
plt.ylabel('Percentage')
plt.xticks(rotation=0)
plt.legend(title='Satisfaction')
plt.tight_layout()
plt.show()

Now, we will handle missing values and encode categorical variables.

In [ ]:
# Handle missing values
print("Handling missing values...")
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median())

# Encode categorical variables
print("Encoding categorical variables...")
df['Gender'] = df['Gender'].map({'Female': 0, 'Male': 1})
df['Customer Type'] = df['Customer Type'].map({'Loyal Customer': 1, 'disloyal Customer': 0})
df['Type of Travel'] = df['Type of Travel'].map({'Personal Travel': 0, 'Business travel': 1})
df['satisfaction_mapped'] = df['satisfaction'].map({'neutral or dissatisfied': 0, 'satisfied': 1})

class_mapping = {'Eco': 0, 'Eco Plus': 1, 'Business': 2}
df['Class'] = df['Class'].map(class_mapping)

print("Preprocessing complete.")
display(df.head())

# Part 2: Unsupervised Learning - Customer Segmentation

We will use K-Means clustering to identify different types of passengers based on their service ratings. First, we need to prepare the data by scaling the relevant features.

In [ ]:
# Select service-related features
service_columns = ['Inflight wifi service', 'Departure/Arrival time convenient',
                   'Ease of Online booking', 'Gate location', 'Food and drink',
                   'Online boarding', 'Seat comfort', 'Inflight entertainment',
                   'On-board service', 'Leg room service', 'Baggage handling',
                   'Checkin service', 'Inflight service', 'Cleanliness']

# Ensure df exists and create X_scaled
if 'df' in globals():
    X_service = df[service_columns]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_service)
    print('Features scaled. X_scaled is now ready for clustering.')
else:
    print('Error: df is not defined. Please execute the first code cell.')

### Elbow Method to find the optimal k
We'll plot the inertia (WCSS) for a range of k values to find the 'elbow' point.

In [ ]:
wcss = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(K_range, wcss, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS')
plt.grid(True)
plt.show()

Based on the elbow plot, let's proceed with $k=3$ (or a value you find most appropriate) to segment the passengers and analyze their characteristics.

In [ ]:
# Fit K-Means with k=3 (standard choice based on elbow behavior in this dataset)
k_chosen = 3
kmeans = KMeans(n_clusters=k_chosen, init='k-means++', random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Compute mean values for each cluster
cluster_profiles = df.groupby('Cluster')[service_columns].mean()

print("Cluster Service Profiles (Mean Ratings):")
display(cluster_profiles)

# Analyze satisfaction distribution per cluster
satisfaction_dist = pd.crosstab(df['Cluster'], df['satisfaction'], normalize='index') * 100
print("\nSatisfaction Distribution (%) by Cluster:")
display(satisfaction_dist)

### Cluster Naming and Interpretation

Based on the mean ratings above, we can name the clusters:
1. **Cluster 0: The Dissatisfied/Low-Rating Group** - Low scores across most digital and comfort metrics.
2. **Cluster 1: The Easy-Going / Average Group** - Moderate scores across the board.
3. **Cluster 2: The Delighted High-Flyers** - Consistently high ratings across all service categories, likely corresponding to high satisfaction rates.

#Part 3: Supervised Learning - Satisfaction Prediction


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from xgboost import XGBClassifier
import numpy as np


# add clusters to feature_columns
feature_columns = service_columns + ['Cluster']
X = df[feature_columns]
y = df['satisfaction_mapped']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Features used    : {feature_columns}\n")


rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

xgb = XGBClassifier(n_estimators=100, random_state=42,
                    eval_metric='logloss', use_label_encoder=False)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)


def get_metrics(name, y_true, y_pred):
    return {
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, pos_label=0), 4), # dissatisfied = 0 is taken as positive label
        'Recall':    round(recall_score(y_true, y_pred, pos_label=0), 4),
        'F1-Score':  round(f1_score(y_true, y_pred, pos_label=0), 4),
    }

results = [
    get_metrics('Random Forest', y_test, y_pred_rf),
    get_metrics('XGBoost',       y_test, y_pred_xgb),
]

results_df = pd.DataFrame(results).set_index('Model')
display(results_df)

# --- Best model by Recall ---
best_model_name = results_df['Recall'].idxmax()
best_model      = rf if best_model_name == 'Random Forest' else xgb
print(f"\nBest model by Recall: {best_model_name}")
print(f"   Recall Score: {results_df.loc[best_model_name, 'Recall']}")


# --- Grouped bar chart ---
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x       = np.arange(len(metrics))
width   = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, results_df.loc['Random Forest', metrics], width, label='Random Forest', color='steelblue')
bars2 = ax.bar(x + width/2, results_df.loc['XGBoost', metrics],       width, label='XGBoost',       color='coral')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

ax.axvspan(2 - 0.4, 2 + 0.4, alpha=0.1, color='green', label='Most Important (Recall)')
ax.set_title('Model Comparison — Random Forest vs XGBoost', fontsize=14, fontweight='bold')
ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.08)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

Metric Focus: RECALL
---------------------
Our goal is to identify DISSATISFIED passengers (class 0).

- A False Negative = we predicted "satisfied" but passenger was actually dissatisfied
- This means the airline takes NO action → passenger churns → revenue lost forever

- A False Positive = we flagged someone as dissatisfied but they were actually fine
- Worst case: airline reaches out unnecessarily → minor cost, no churn

Missing a dissatisfied customer is FAR more costly than over-flagging a satisfied one.
Therefore RECALL is the most critical metric — we want to catch as many
dissatisfied passengers as possible, even at the cost of some precision.

#Part 4: Actionable Business Strategy


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

feature_names         = X_train.columns.tolist()
importances           = best_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature':    feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

top3_features = feature_importance_df.head(3)['Feature'].tolist()
print("Top 3 Features driving satisfaction:")
print(top3_features)

top10  = feature_importance_df.head(10)
colors = ['crimson' if f in top3_features else 'steelblue' for f in top10['Feature'][::-1]]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10['Feature'][::-1], top10['Importance'][::-1], color=colors)
ax.set_title(f'Top 10 Feature Importances — {best_model_name}', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("Red bars = Top 3 most impactful features on satisfaction")

cluster_satisfaction = df.groupby('Cluster')['satisfaction_mapped'].mean()
worst_cluster        = cluster_satisfaction.idxmin()

print(f"\nWorst Cluster: Cluster {worst_cluster}")
print(f"   Satisfaction Rate: {cluster_satisfaction[worst_cluster]:.2%}\n")

worst_df = df[df['Cluster'] == worst_cluster]

class_map  = {0: 'Eco', 1: 'Eco Plus', 2: 'Business'}
ctype_map  = {1: 'Loyal', 0: 'Disloyal'}
travel_map = {0: 'Personal', 1: 'Business'}

class_dist  = worst_df['Class'].map(class_map).value_counts(normalize=True) * 100
ctype_dist  = worst_df['Customer Type'].map(ctype_map).value_counts(normalize=True) * 100
travel_dist = worst_df['Type of Travel'].map(travel_map).value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f'Cluster {worst_cluster} — Passenger Profile (Worst Cluster)',
             fontsize=14, fontweight='bold')

for ax, dist, title, color in zip(
    axes,
    [class_dist, ctype_dist, travel_dist],
    ['Class Distribution', 'Customer Type', 'Type of Travel'],
    ['coral', 'steelblue', 'mediumseagreen']
):
    ax.bar(dist.index, dist.values, color=color)
    ax.set_title(title)
    ax.set_ylabel('Percentage (%)')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"""

         BUSINESS RECOMMENDATIONS — CLUSTER {worst_cluster}


Cluster {worst_cluster} has the highest proportion of dissatisfied passengers.
Key profile: mostly {travel_dist.idxmax()} travellers, {class_dist.idxmax()} class,
             {ctype_dist.idxmax()} customers.

Based on the top 3 features driving satisfaction:

1️.  {top3_features[0]}
    → Invest directly in improving this service for Cluster {worst_cluster}.
      Even marginal improvements here has the highest impact on retention.

2️.  {top3_features[1]}
    → This is the second biggest lever. Focus operational improvements
      or staff training specifically targeting this pain point.

3️.  {top3_features[2]}
    → Address this for {class_dist.idxmax()} class {travel_dist.idxmax()} travellers
      specifically — they are the dominant group in this cluster.

Overall: Retaining Cluster {worst_cluster} passengers requires targeted service
improvements in the above areas. A personalised outreach campaign
for disloyal passengers in this cluster could significantly
reduce churn risk.
""")

## Conclusion

This analysis reveals that **56.67% of airline passengers are dissatisfied** — a majority — making retention a critical business priority.

Through clustering, three distinct passenger segments emerged. **Cluster 0**, the worst-performing group with an 86.3% dissatisfaction rate, was identified as the primary target for intervention. Profiling this cluster revealed a counterintuitive finding: these are predominantly **loyal business travellers flying Eco class** — frequent flyers who have not yet churned but are deeply underserved.

The XGBoost classifier outperformed Random Forest across all metrics, achieving a **Recall of 96.54%** — meaning it correctly identifies nearly all dissatisfied passengers, minimising the risk of missed interventions. Feature importance analysis confirmed that **Online boarding** and **Inflight wifi service** are the two strongest drivers of satisfaction, followed by cluster membership itself — indicating that dissatisfaction in Cluster 0 is systemic rather than individual.

**The core business recommendation** is clear: the airline should prioritise improving the digital experience — specifically online boarding and inflight wifi — with targeted upgrades for Eco class business travellers. A loyalty programme redesign offering wifi credits and boarding perks for this segment could convert passive loyalty into genuine satisfaction before churn occurs.

These findings demonstrate that machine learning can move beyond prediction into prescriptive strategy — not just identifying who is at risk, but telling the airline exactly where to act and for whom.